# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madihakomal75/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Rule Definition & Decision Logic

We compute a deterministic baseline action score using three core page health metrics: staleness, high historical impression volume, and ranking position collapse.

**Scoring Logic:**
* `action_score` = $\text{staleness\_flag} \times \text{high\_volume\_flag} \times \text{impressions\_90d}$

**Reason Codes Assigned:**
1. **`HIGH_IMP_STALE`**: Page has high historical visibility ($\ge 500$ 90-day impressions) and hasn't been updated in over 180 days (`days_since_last_update >= 180`).
2. **`RANK_DROPPED`**: Average ranking position fell beyond position 15 (`avg_position > 15`) despite historic impressions.
3. **`LOW_TRAFFIC_FRESH`**: Page is updated recently or has negligible traffic ($\text{impressions\_90d} < 500$).

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/madihakomal75/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np

# Load raw dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Ensure output directory exists
os.makedirs("work/outputs", exist_ok=True)

print(f"Dataset loaded. Total rows: {len(df):,}")

Dataset loaded. Total rows: 30,000


### Baseline Score Calculation & Queue Export

The scoring script evaluates all candidate pages, calculates the priority score, sorts pages in descending order of priority score, and exports the final queue to `work/outputs/baseline_action_score.csv`.

In [2]:
# Identify ID column dynamically
id_col = "url_hash_id" if "url_hash_id" in df.columns else ("page_id" if "page_id" in df.columns else df.columns[0])

# Define Rule Flags
is_stale = (df["days_since_last_update"] >= 180).astype(int)
is_high_volume = (df["impressions_90d"] >= 500).astype(int)

# Compute Score
df["baseline_action_score"] = is_stale * is_high_volume * df["impressions_90d"]

# Assign Reason Codes
def assign_reason(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "HIGH_IMP_STALE"
    elif row["avg_position"] > 15 and row["impressions_90d"] >= 500:
        return "RANK_DROPPED"
    else:
        return "LOW_TRAFFIC_FRESH"

df["reason_code"] = df.apply(assign_reason, axis=1)

# Rank pages (1 = highest priority)
df_ranked = df.sort_values(by="baseline_action_score", ascending=False).reset_index(drop=True)
df_ranked["priority_rank"] = df_ranked.index + 1

# Save output CSV
output_cols = [id_col, "priority_rank", "baseline_action_score", "reason_code", "days_since_last_update", "impressions_90d", "avg_position"]
df_ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Exported {len(df_ranked):,} ranked rows to work/outputs/baseline_action_score.csv")
df_ranked[output_cols].head(5)

Exported 30,000 ranked rows to work/outputs/baseline_action_score.csv


,content_id,priority_rank,baseline_action_score,reason_code,days_since_last_update,impressions_90d,avg_position
0,content_cf56e2e2e282,1,61678,HIGH_IMP_STALE,194,61678,19.7
1,content_7368877ea310,2,59472,HIGH_IMP_STALE,194,59472,24.8
2,content_1bfaa38ff26c,3,25715,HIGH_IMP_STALE,194,25715,22.2
3,content_0a91db491d14,4,13299,HIGH_IMP_STALE,193,13299,10.5
4,content_5feee3994adb,5,7812,HIGH_IMP_STALE,194,7812,39.0


### Qualitative Review of Top-20 Priority Queue

We inspect the top 20 pages output by the baseline action rule to verify whether flagged URLs genuinely require content refreshes or present false positive risks.

In [3]:
# Display top 20 review candidate slice
top_20 = df_ranked.head(20)[[id_col, "priority_rank", "baseline_action_score", "reason_code", "days_since_last_update", "impressions_90d", "avg_position"]]

print("Top 20 Ranked Pages Audit:")
top_20

Top 20 Ranked Pages Audit:


,content_id,priority_rank,baseline_action_score,reason_code,days_since_last_update,impressions_90d,avg_position
0,content_cf56e2e2e282,1,61678,HIGH_IMP_STALE,194,61678,19.7
1,content_7368877ea310,2,59472,HIGH_IMP_STALE,194,59472,24.8
2,content_1bfaa38ff26c,3,25715,HIGH_IMP_STALE,194,25715,22.2
3,content_0a91db491d14,4,13299,HIGH_IMP_STALE,193,13299,10.5
4,content_5feee3994adb,5,7812,HIGH_IMP_STALE,194,7812,39.0
5,content_c2d929d83eaa,6,7558,HIGH_IMP_STALE,193,7558,17.9
6,content_b16bd7307b39,7,4590,HIGH_IMP_STALE,194,4590,31.0
7,content_fe16a55cd13d,8,4556,HIGH_IMP_STALE,194,4556,16.4
8,content_ecb6215e79fd,9,4429,HIGH_IMP_STALE,194,4429,25.3
9,content_928af3e22c80,10,1697,HIGH_IMP_STALE,193,1697,15.8


### Model Limitations & Leakage Verification

1. **Weak Picks Identified:**
   * High-impression pages with excellent rankings (`avg_position <= 3.0`) are included in top recommendations simply because they haven't been edited in 180 days. Reframing an already top-ranked page introduces unnecessary risk of rank volatility.
   * Pages with seasonal spikes can skew impression counts without having actual structural content decay.

2. **Data Leakage Verification:**
   * All rules rely strictly on historical observation windows (`impressions_90d`, `days_since_last_update`).
   * Target outcome indicators (`trend_direction`, `is_declining_label`) were completely omitted during queue generation.

In [5]:
# Audit top 50 ranked pages for low-value / strong-performing pages (potential weak picks)
top_50 = df_ranked.head(50)
weak_picks = top_50[top_50["avg_position"] <= 3.0]

print(f"Weak Picks in Top 50 (Already top-ranked position <= 3.0): {len(weak_picks)} pages")
if len(weak_picks) > 0:
    print(weak_picks[[id_col, "priority_rank", "impressions_90d", "avg_position", "days_since_last_update"]].head(5).to_string())

# Verify target label was not leaked into scoring script
assert "trend_direction" not in df_ranked[["baseline_action_score", "reason_code"]].columns
print("\nLeakage Check Passed: Target label was not used in calculating priority scores.")

Weak Picks in Top 50 (Already top-ranked position <= 3.0): 3 pages
              content_id  priority_rank  impressions_90d  avg_position  days_since_last_update
22  content_6a21f0fe9d71             23                1           0.0                      20
25  content_8cce8bc6aef7             26                1           0.0                      20
44  content_a5f950cbaee0             45             6076           1.8                      20

Leakage Check Passed: Target label was not used in calculating priority scores.
